# as-strided-noncontig-source — ex5: rolling window via as_strided (zero-copy)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `as-strided-noncontig-source`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five patterns around tensor memory layout that ramp from reading strides → recognizing transpose breaks contiguity → seeing `.view()` fail on non-contig → fixing it with `.contiguous()` → building a zero-copy sliding-window view via `as_strided`. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `as-strided-noncontig-source`**, which bridges to the bank subtopic `Numpy: Applied patterns and advanced` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "as-strided-noncontig-source"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Strides and contiguity — quick refresher

**Stride** = number of elements to skip in storage to advance one step along that axis.
- A contiguous `(H, W)` tensor has stride `(W, 1)`.
- A contiguous `(B, C, H, W)` tensor has stride `(C*H*W, H*W, W, 1)`.

**Contiguity** = the strides match the row-major layout of the current shape.
- `.T` swaps strides but not data → the result is a view but not contiguous.
- `.view()` requires contiguous input — it never copies.
- `.reshape()` makes a view if possible, copies if not.
- `.contiguous()` forces a row-major copy if the tensor isn't already contiguous.

**`as_strided(size, stride)`** is the lowest-level view constructor — you provide the exact shape and stride pair. Bypasses all safety checks; trust the values you pass.

### Exercise 5 — rolling window via as_strided (zero-copy)

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize stride arithmetic with `as_strided` to build an O(1)-memory sliding-window view of a 1-D tensor.
> Keywords: as-strided, sliding-window, integration, multi-kc
> ```

**KCs targeted:** `strides-anatomy`, `as-strided-rolling-window`

Implement `ex5_rolling_window(x, w)` to build a 2-D view where row `i` is the length-`w` window starting at position `i` of `x`.

Input shape: `(N,)` (1-D). Output shape: `(N - w + 1, w)`. Each `out[i, j] == x[i + j]`.

Use `x.as_strided(size, stride)` directly — no Python loop, no `.clone()`, no `torch.cat`. The result must share storage with `x` (it's a view).

**Stride hint:** moving one step along the output's row axis = moving one step along the source. Moving one step along the output's column axis = also moving one step along the source. So both output strides equal `x.stride(0)`.

> ⚠️ **Integrative exercise.** This combines stride mechanics + memory aliasing + the as_strided API surface — empirical work (Lohr et al. ITiCSE 2025) shows 3-concept LLM-generated exercises drop from ~94% to ~40% solvability. Expect a step in difficulty here vs Exercises 1-4.

In [ ]:
def ex5_rolling_window(x: Tensor, w: int) -> Tensor:
    """Sliding window over a 1-D tensor. (N,) → (N-w+1, w) view."""
    raise NotImplementedError()


def _test_ex5():
    x = t.arange(7).float()                 # [0, 1, 2, 3, 4, 5, 6]
    y = ex5_rolling_window(x, w=3)
    assert y.shape == (5, 3), f'expected (5,3), got {tuple(y.shape)}'
    expected = t.tensor([
        [0., 1., 2.],
        [1., 2., 3.],
        [2., 3., 4.],
        [3., 4., 5.],
        [4., 5., 6.],
    ])
    assert t.equal(y, expected), f'value mismatch:\n{y}'
    # Must be a view, not a copy — same storage pointer.
    assert y.data_ptr() == x.data_ptr(), 'rolling-window result should share storage with x (zero-copy view)'
    _dd_passed.add('ex5')
    print("ex5 ✓")

_test_ex5()

<details><summary>Solution</summary>

```python
def ex5_rolling_window(x: Tensor, w: int) -> Tensor:
    n = x.shape[0]
    s = x.stride(0)
    return x.as_strided(size=(n - w + 1, w), stride=(s, s))
```

**Reading the stride pair.**
- Output stride for axis 0 = `s` (step from row `i` to row `i+1` = advance one element in source).
- Output stride for axis 1 = `s` (step from column `j` to column `j+1` within a row = also advance one element).

Both row and column advance one source-element. The 2-D view **overlaps** itself — every source element appears in multiple rows. That's the magic: O(1) memory, no copy.

**Danger zone.** `as_strided` does NOT check bounds. If you pass a stride/size that walks off the end of storage, you get undefined behavior — silent garbage, or a segfault. Use the higher-level `torch.nn.functional.unfold` / `torch.tensor.unfold(dim, size, step)` for sliding windows in production code; this exercise is to understand what those calls do under the hood.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex5'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex5',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()